In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8 as required
tz = pytz.timezone('Asia/Shanghai')

# load files (stateful variables for reuse)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# metric.csv summary
metric_rows = df_metric.shape[0]
metric_cmdb_ids = list(pd.unique(df_metric['cmdb_id']))[:50]
metric_kpis = list(pd.unique(df_metric['kpi_name']))[:50]
metric_ts = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metric_min_ts = metric_ts.min()
metric_max_ts = metric_ts.max()
metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'rows': metric_rows,
    'unique_cmdb_ids_count': len(metric_cmdb_ids),
    'unique_cmdb_ids_sample': metric_cmdb_ids,
    'unique_kpi_count': len(metric_kpis),
    'unique_kpi_sample': metric_kpis,
    'min_timestamp': metric_min_ts,
    'max_timestamp': metric_max_ts
}])

# trace.csv summary
trace_rows = df_trace.shape[0]
trace_cmdb_ids = list(pd.unique(df_trace['cmdb_id']))[:50]
trace_names = list(pd.unique(df_trace['trace_name']))[:50]
trace_ts = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_min_ts = trace_ts.min()
trace_max_ts = trace_ts.max()
trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'rows': trace_rows,
    'unique_cmdb_ids_count': len(trace_cmdb_ids),
    'unique_cmdb_ids_sample': trace_cmdb_ids,
    'unique_trace_count': len(trace_names),
    'unique_trace_sample': trace_names,
    'min_timestamp': trace_min_ts,
    'max_timestamp': trace_max_ts
}])

# Display compact summaries (two variables as required)
metric_summary, trace_summary

```
Out[1]:
```
import pandas as pd

# reuse existing summary DataFrames from the kernel if available
try:
    ms = metric_summary.loc[0]
    ts = trace_summary.loc[0]
except NameError:
    # fallback: reload files and recreate minimal summaries
    df_metric = pd.read_csv('metric.csv')
    df_trace = pd.read_csv('trace.csv')
    ms = {
        'rows': df_metric.shape[0],
        'unique_cmdb_ids_sample': list(pd.unique(df_metric['cmdb_id']))[:50],
        'unique_kpi_sample': list(pd.unique(df_metric['kpi_name']))[:50],
        'min_timestamp': pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Shanghai').min(),
        'max_timestamp': pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Shanghai').max()
    }
    ts = {
        'rows': df_trace.shape[0],
        'unique_cmdb_ids_sample': list(pd.unique(df_trace['cmdb_id']))[:50],
        'unique_trace_sample': list(pd.unique(df_trace['trace_name']))[:50],
        'min_timestamp': pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Shanghai').min(),
        'max_timestamp': pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Shanghai').max()
    }

summary = (
    f"metric.csv: rows={ms['rows']}; "
    f"unique cmdb_id sample (up to 50): {ms['unique_cmdb_ids_sample']}; "
    f"unique kpi sample (up to 50): {ms['unique_kpi_sample']}; "
    f"timestamp range: {ms['min_timestamp']} to {ms['max_timestamp']}\n\n"
    f"trace.csv: rows={ts['rows']}; "
    f"unique cmdb_id sample (up to 50): {ts['unique_cmdb_ids_sample']}; "
    f"unique trace sample (up to 50): {ts.get('unique_trace_sample', ts.get('unique_trace_sample'))}; "
    f"timestamp range: {ts['min_timestamp']} to {ts['max_timestamp']}"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file   rows  unique_cmdb_ids_count                             unique_cmdb_ids_sample  unique_kpi_count                                  unique_kpi_sample             min_timestamp             max_timestamp
0  metric.csv  90687                     50  [db_001, db_002, db_003, db_004, db_005, db_00...                50  [ACS, AIOS, AWS, CPU_Used_Pct, CPU_free_pct, C... 2020-05-31 02:00:00+08:00 2020-05-31 03:00:00+08:00,         file  rows  unique_cmdb_ids_count                             unique_cmdb_ids_sample  unique_trace_count                                unique_trace_sample             min_timestamp             max_timestamp
0  trace.csv  6240                     10  [docker_001, docker_002, docker_003, docker_00...                  50  [trace.from_os_022.remoteprocess.duration_mean... 2020-05-31 02:30:00+08:00 2020-05-31 02:59:00+08:00)```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# compute global statistics per (cmdb_id, kpi_name) using entire file
metric_stats = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count='count',
         p95=lambda x: x.quantile(0.95),
         min='min',
         max='max',
         mean='mean')
    .reset_index()
)

# sort by p95 descending and keep top 20 rows, selected columns/order
metric_stats_top = metric_stats.sort_values('p95', ascending=False)[
    ['cmdb_id', 'kpi_name', 'count', 'p95', 'min', 'max', 'mean']
].head(20)

metric_stats_top

```
Out[2]:
```
import pandas as pd

# reuse metric_stats_top from the kernel if available
try:
    top = metric_stats_top.head(20)
except NameError:
    # fallback: recompute if not present
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    metric_stats = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(count='count',
             p95=lambda x: x.quantile(0.95),
             min='min',
             max='max',
             mean='mean')
        .reset_index()
    )
    top = metric_stats.sort_values('p95', ascending=False)[
        ['cmdb_id', 'kpi_name', 'count', 'p95', 'min', 'max', 'mean']
    ].head(20)

# Build a concise plain-English summary using the top rows
lines = []
lines.append("Summary of top 20 (by P95) for each (cmdb_id, kpi_name):")
for i, row in top.head(5).iterrows():  # include the top 5 as representative highlights
    lines.append(
        f"- {row['cmdb_id']} / {row['kpi_name']}: P95={int(row['p95']):,}, "
        f"count={int(row['count'])}, min={int(row['min']):,}, max={int(row['max']):,}, mean={int(row['mean']):,}"
    )

# Add an overall observation
lines.append("")
lines.append("Overall observation: The top entries are dominated by Redis metrics (maxmemory, keyspace_hits/misses, used_memory and peaks). "
             "P95 values are very large (billions), with counts typically 59–60. This indicates the highest-value KPIs in the dataset are Redis memory and keyspace counters.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          kpi_name  count           p95           min           max          mean
1859  redis_004         maxmemory     59  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09
1947  redis_008         maxmemory     59  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09
2035  redis_012         maxmemory     60  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09
1791  redis_001     keyspace_hits     60  5.949096e+09  5.942463e+09  5.949447e+09  5.945954e+09
1813  redis_002     keyspace_hits     60  5.895730e+09  5.889152e+09  5.896076e+09  5.892614e+09
1967  redis_009     keyspace_hits     60  5.836734e+09  5.830226e+09  5.837081e+09  5.833651e+09
1814  redis_002   keyspace_misses     60  4.583303e+09  4.578278e+09  4.583569e+09  4.580923e+09
1792  redis_001   keyspace_misses     60  4.488978e+09  4.484052e+09  4.489238e+09  4.486644e+09
1968  redis_009   keyspace_misses     60  4.474464e+09  4.469559e+09  4.474723e+09  4.472141e+09
1869  redis_004  used_memory_peak     59  3.027600e+09  3.027600e+09  3.027600e+09  3.027600e+09
2045  redis_012  used_memory_peak     60  3.006680e+09  3.006680e+09  3.006680e+09  3.006680e+09
1957  redis_008  used_memory_peak     59  2.945528e+09  2.945528e+09  2.945528e+09  2.945528e+09
1956  redis_008       used_memory     59  2.738627e+09  2.738144e+09  2.738697e+09  2.738365e+09
1868  redis_004       used_memory     59  2.738423e+09  2.737907e+09  2.738460e+09  2.738231e+09
2044  redis_012       used_memory     60  2.738421e+09  2.738163e+09  2.738421e+09  2.738340e+09
1870  redis_004   used_memory_rss     59  2.564432e+09  2.564432e+09  2.564432e+09  2.564432e+09
1958  redis_008   used_memory_rss     59  2.555433e+09  2.555433e+09  2.555433e+09  2.555433e+09
2046  redis_012   used_memory_rss     60  2.544013e+09  2.544013e+09  2.544013e+09  2.544013e+09
1903  redis_006         maxmemory     60  2.000000e+09  2.000000e+09  2.000000e+09  2.000000e+09
1837  redis_003         maxmemory     59  2.000000e+09  2.000000e+09  2.000000e+09  2.000000e+09```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure numeric value
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# compute or reuse global P95 per (cmdb_id, kpi_name) BEFORE any time filtering
try:
    metric_global_p95 = metric_stats.reset_index(drop=True)[['cmdb_id', 'kpi_name', 'p95']].rename(columns={'p95':'global_p95'})
except Exception:
    metric_global_p95 = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# create timezone-aware timestamp column in UTC+8
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define incident window (inclusive start, exclusive end) in UTC+8
start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)

# filter rows to window
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# merge global p95 into windowed rows
df_window = df_window.merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# detect breaches: value > global_p95
df_window['is_breach'] = df_window['value'] > df_window['global_p95']

# compute count_of_breaches and earliest breach timestamp from breach rows
breach_rows = df_window[df_window['is_breach']].copy()
breach_agg = (
    breach_rows
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(count_of_breaches=('is_breach', 'sum'),
         earliest_breach_timestamp=('ts', 'min'))
    .reset_index()
)

# compute max_value_in_window per pair (using all window rows for the pair)
max_window = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .max()
    .reset_index(name='max_value_in_window')
)

# assemble results for pairs that had at least one breach
result = (
    breach_agg
    .merge(max_window, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')
)

# compute severity safely (handle zero global_p95)
result['severity'] = np.where(
    result['global_p95'] > 0,
    (result['max_value_in_window'] - result['global_p95']) / result['global_p95'],
    np.nan
)

# sort by severity desc and keep top 20
result_sorted = result.sort_values('severity', ascending=False)[
    ['cmdb_id', 'kpi_name', 'count_of_breaches', 'earliest_breach_timestamp',
     'max_value_in_window', 'global_p95', 'severity']
].head(20)

# keep output compact: round numeric columns for readability
result_sorted['max_value_in_window'] = result_sorted['max_value_in_window'].round(6)
result_sorted['global_p95'] = result_sorted['global_p95'].round(6)
result_sorted['severity'] = result_sorted['severity'].round(6)

# final variable to display
result_sorted

```
Out[3]:
```
import pandas as pd

# reuse result and result_sorted from kernel if available
try:
    total_breached_pairs = int(result.shape[0])
    top_df = result_sorted.copy()
except NameError:
    # fallback: load and compute minimal info (should not be needed normally)
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    metric_global_p95 = (
        df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95)).reset_index()
    )
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Shanghai')
    start = pd.Timestamp('2020-05-31 02:30:00').tz_localize('Asia/Shanghai')
    end = pd.Timestamp('2020-05-31 03:00:00').tz_localize('Asia/Shanghai')
    df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')
    df_window['is_breach'] = df_window['value'] > df_window['global_p95']
    breach_rows = df_window[df_window['is_breach']].copy()
    breach_agg = breach_rows.groupby(['cmdb_id','kpi_name']).agg(
        count_of_breaches=('is_breach','sum'),
        earliest_breach_timestamp=('ts','min')
    ).reset_index()
    max_window = df_window.groupby(['cmdb_id','kpi_name'])['value'].max().reset_index(name='max_value_in_window')
    result = breach_agg.merge(max_window, on=['cmdb_id','kpi_name']).merge(metric_global_p95, on=['cmdb_id','kpi_name'])
    result['severity'] = (result['max_value_in_window'] - result['global_p95']) / result['global_p95']
    total_breached_pairs = int(result.shape[0])
    top_df = result.sort_values('severity', ascending=False)[
        ['cmdb_id','kpi_name','count_of_breaches','earliest_breach_timestamp','max_value_in_window','global_p95','severity']
    ].head(20)

# Build a concise plain-English summary using top entries (up to 5)
top_n = top_df.head(5)
lines = []
lines.append(f"Incident window: 2020-05-31 02:30:00+08 to 2020-05-31 03:00:00+08.")
lines.append(f"Number of (cmdb_id, kpi_name) pairs that breached their global P95: {total_breached_pairs}.")
lines.append("Top breaches by severity (up to 5):")
for _, r in top_n.iterrows():
    ts = r['earliest_breach_timestamp']
    ts_str = pd.to_datetime(ts).tz_convert('Asia/Shanghai').isoformat() if pd.notnull(ts) else "N/A"
    lines.append(
        f"- {r['cmdb_id']} / {r['kpi_name']}: breaches={int(r['count_of_breaches'])}, "
        f"earliest={ts_str}, max_in_window={float(r['max_value_in_window']):,}, "
        f"global_p95={float(r['global_p95']):,}, severity={float(r['severity']):.3f}"
    )

lines.append("")
lines.append("Observation: Top offenders are database metrics (e.g., DB write/read rates and LFSync/LFParaWrite) and OS disk latency (Disk_await), plus one container CPU metric. These show large relative excursions above their historical P95s during the incident window.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name  count_of_breaches earliest_breach_timestamp  max_value_in_window   global_p95    severity
177      db_011        LFParaWrite_Per_Sec                  1 2020-05-31 02:51:00+08:00               17.890     0.140000  126.785714
541      os_018                 Disk_await                  3 2020-05-31 02:42:00+08:00               56.750     0.638000   87.949843
178      db_011             LFSync_Per_Sec                  2 2020-05-31 02:37:00+08:00               37.990     0.451000   83.235033
27       db_002      Physical_Read_Per_Sec                  2 2020-05-31 02:38:00+08:00               16.900     0.205000   81.439024
23       db_002         Logic_Read_Per_Sec                  2 2020-05-31 02:44:00+08:00            16986.100   282.395000   59.150144
721   redis_010  instantaneous_output_kbps                  1 2020-05-31 02:57:00+08:00                2.960     0.060000   48.333333
151      db_009               Redo_Per_Sec                  2 2020-05-31 02:38:00+08:00            57149.100  3154.280000   17.117954
201      db_012            SeqRead_Per_Sec                  2 2020-05-31 02:43:00+08:00                2.570     0.191500   12.420366
582      os_020                 Disk_await                  1 2020-05-31 02:31:00+08:00               48.850     5.097500    8.583129
50       db_003                Session_pct                  2 2020-05-31 02:34:00+08:00                8.713     0.949000    8.181243
36       db_003                     DbTime                  2 2020-05-31 02:34:00+08:00             2552.600   280.275000    8.107484
48       db_003                Sess_Active                  2 2020-05-31 02:34:00+08:00               25.500     2.835000    7.994709
227  docker_008         container_cpu_used                  1 2020-05-31 02:54:00+08:00               27.000     3.050000    7.852459
208      db_013        DFParaWrite_Per_Sec                  2 2020-05-31 02:40:00+08:00                0.530     0.070500    6.517730
24       db_002              Login_Per_Sec                  1 2020-05-31 02:44:00+08:00                2.400     0.405000    4.925926
438      os_012                 Disk_await                  1 2020-05-31 02:57:00+08:00               32.750     5.659000    4.787242
608      os_021                 Disk_svctm                  2 2020-05-31 02:32:00+08:00                3.660     0.709000    4.162200
527      os_017                 Disk_svctm                  2 2020-05-31 02:47:00+08:00               13.700     2.686000    4.100521
260      os_003                 Disk_await                  3 2020-05-31 02:52:00+08:00               11.200     2.841500    2.941580
346      os_006       Processor_load_1_min                  2 2020-05-31 02:33:00+08:00                0.025     0.006834    2.658394```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse timezone and data if available, otherwise load/compute
tz = pytz.timezone('Asia/Shanghai')

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure numeric and ts columns
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# compute or reuse global P95 per (cmdb_id, kpi_name) BEFORE filtering
try:
    metric_global_p95
except NameError:
    metric_global_p95 = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# incident window bounds (inclusive start, exclusive end) in UTC+8
start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)

# filter to window
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# merge global p95
df_window = df_window.merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# identify breach rows (value > global_p95)
breach_rows = df_window[df_window['value'] > df_window['global_p95']].copy()

# if no breaches, return empty results
if breach_rows.shape[0] == 0:
    runs_top = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_length_in_minutes','start_timestamp','end_timestamp',
        'max_value_in_run','global_p95','severity'
    ])
    total_runs_after_filter = 0
else:
    # sort by identifiers and timestamp
    breach_rows = breach_rows.sort_values(['cmdb_id', 'kpi_name', 'ts']).reset_index(drop=True)

    # compute diff in seconds between consecutive breached timestamps within each (cmdb_id,kpi_name)
    breach_rows['prev_ts'] = breach_rows.groupby(['cmdb_id', 'kpi_name'])['ts'].shift(1)
    breach_rows['diff_secs'] = (breach_rows['ts'] - breach_rows['prev_ts']).dt.total_seconds().fillna(np.nan)

    # a new run starts when diff_secs != 60 OR prev_ts is NaN
    breach_rows['new_run'] = ((breach_rows['diff_secs'] != 60) | breach_rows['diff_secs'].isna()).astype(int)

    # assign run_id by cumulative sum of new_run per group of (cmdb_id,kpi_name)
    breach_rows['run_id'] = breach_rows.groupby(['cmdb_id', 'kpi_name'])['new_run'].cumsum()

    # aggregate per run (group keys include cmdb_id,kpi_name,run_id)
    runs = (
        breach_rows
        .groupby(['cmdb_id', 'kpi_name', 'run_id'])
        .agg(
            run_length_in_minutes=('ts', 'count'),
            start_timestamp=('ts', 'min'),
            end_timestamp=('ts', 'max'),
            max_value_in_run=('value', 'max'),
            global_p95=('global_p95', 'first')
        )
        .reset_index(drop=False)
    )

    # compute severity safely
    runs['severity'] = np.where(
        runs['global_p95'] > 0,
        (runs['max_value_in_run'] - runs['global_p95']) / runs['global_p95'],
        np.nan
    )

    # filter runs: keep those with run_length_in_minutes >=2 OR severity > 0.5
    runs_filtered = runs[(runs['run_length_in_minutes'] >= 2) | (runs['severity'] > 0.5)].copy()

    # sort by severity desc, keep top 20
    runs_top = runs_filtered.sort_values('severity', ascending=False)[
        ['cmdb_id','kpi_name','run_length_in_minutes','start_timestamp','end_timestamp',
         'max_value_in_run','global_p95','severity']
    ].head(20).reset_index(drop=True)

    # round numeric columns for compactness
    runs_top['max_value_in_run'] = runs_top['max_value_in_run'].round(6)
    runs_top['global_p95'] = runs_top['global_p95'].round(6)
    runs_top['severity'] = runs_top['severity'].round(6)

    total_runs_after_filter = int(runs_filtered.shape[0])

# final compact outputs
runs_top, total_runs_after_filter

```
Out[4]:
```
import pandas as pd

# reuse runs_top and total_runs_after_filter if available
try:
    top_df = runs_top.copy()
    total = total_runs_after_filter
except NameError:
    # fallback: minimal message if variables are not in kernel
    summary = "No run results available in the current session."
    summary

# build plain-English summary string
lines = []
lines.append("Incident window: 2020-05-31 02:30:00+08 to 2020-05-31 03:00:00+08.")
lines.append(f"Total number of breach runs after filtering: {total}.")
lines.append("Top runs by severity (up to 5):")

for _, r in top_df.head(5).iterrows():
    start = pd.to_datetime(r['start_timestamp']).tz_convert('Asia/Shanghai').isoformat()
    end = pd.to_datetime(r['end_timestamp']).tz_convert('Asia/Shanghai').isoformat()
    lines.append(
        f"- {r['cmdb_id']} / {r['kpi_name']}: run_length={int(r['run_length_in_minutes'])} min, "
        f"start={start}, end={end}, max_in_run={float(r['max_value_in_run']):,}, "
        f"global_p95={float(r['global_p95']):,}, severity={float(r['severity']):.3f}"
    )

lines.append("")
lines.append("Observation: Most high-severity runs are database-related KPIs (e.g., LFParaWrite_Per_Sec, LFSync_Per_Sec, read rates) and OS disk latency (Disk_await). A notable multi-minute run is os_018 Disk_await (2 minutes) with very large severity; many other runs are short (1 minute) but have large relative excursions above their historical P95s.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                   kpi_name  run_length_in_minutes           start_timestamp             end_timestamp  max_value_in_run  global_p95    severity
0       db_011        LFParaWrite_Per_Sec                      1 2020-05-31 02:51:00+08:00 2020-05-31 02:51:00+08:00            17.890      0.1400  126.785714
1       os_018                 Disk_await                      2 2020-05-31 02:42:00+08:00 2020-05-31 02:43:00+08:00            56.750      0.6380   87.949843
2       db_011             LFSync_Per_Sec                      1 2020-05-31 02:51:00+08:00 2020-05-31 02:51:00+08:00            37.990      0.4510   83.235033
3       db_002      Physical_Read_Per_Sec                      1 2020-05-31 02:58:00+08:00 2020-05-31 02:58:00+08:00            16.900      0.2050   81.439024
4       db_002         Logic_Read_Per_Sec                      1 2020-05-31 02:44:00+08:00 2020-05-31 02:44:00+08:00         16986.100    282.3950   59.150144
5    redis_010  instantaneous_output_kbps                      1 2020-05-31 02:57:00+08:00 2020-05-31 02:57:00+08:00             2.960      0.0600   48.333333
6       os_018                 Disk_await                      1 2020-05-31 02:54:00+08:00 2020-05-31 02:54:00+08:00            12.760      0.6380   19.000000
7       db_009               Redo_Per_Sec                      1 2020-05-31 02:59:00+08:00 2020-05-31 02:59:00+08:00         57149.100   3154.2800   17.117954
8       db_012            SeqRead_Per_Sec                      1 2020-05-31 02:49:00+08:00 2020-05-31 02:49:00+08:00             2.570      0.1915   12.420366
9       os_020                 Disk_await                      1 2020-05-31 02:31:00+08:00 2020-05-31 02:31:00+08:00            48.850      5.0975    8.583129
10      db_003                Session_pct                      1 2020-05-31 02:39:00+08:00 2020-05-31 02:39:00+08:00             8.713      0.9490    8.181243
11      db_003                     DbTime                      1 2020-05-31 02:39:00+08:00 2020-05-31 02:39:00+08:00          2552.600    280.2750    8.107484
12      db_003                Sess_Active                      1 2020-05-31 02:39:00+08:00 2020-05-31 02:39:00+08:00            25.500      2.8350    7.994709
13  docker_008         container_cpu_used                      1 2020-05-31 02:54:00+08:00 2020-05-31 02:54:00+08:00            27.000      3.0500    7.852459
14      db_013        DFParaWrite_Per_Sec                      1 2020-05-31 02:40:00+08:00 2020-05-31 02:40:00+08:00             0.530      0.0705    6.517730
15      db_002              Login_Per_Sec                      1 2020-05-31 02:44:00+08:00 2020-05-31 02:44:00+08:00             2.400      0.4050    4.925926
16      os_012                 Disk_await                      1 2020-05-31 02:57:00+08:00 2020-05-31 02:57:00+08:00            32.750      5.6590    4.787242
17      os_021                 Disk_svctm                      1 2020-05-31 02:45:00+08:00 2020-05-31 02:45:00+08:00             3.660      0.7090    4.162200
18      os_017                 Disk_svctm                      1 2020-05-31 02:52:00+08:00 2020-05-31 02:52:00+08:00            13.700      2.6860    4.100521
19      os_003                 Disk_await                      1 2020-05-31 02:59:00+08:00 2020-05-31 02:59:00+08:00            11.200      2.8415    2.941580, 178)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# ensure numeric value column
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Compute global P95 per trace_name using the entire file (before any filtering)
global_p95_trace = (
    df_trace
    .groupby('trace_name')['value']
    .agg(global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# create timezone-aware timestamp column in UTC+8
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define incident window (inclusive start, exclusive end)
start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)

# filter rows to the window
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].copy()

# merge global p95 (by trace_name)
df_trace_window = df_trace_window.merge(global_p95_trace, on='trace_name', how='left')

# detect breaches: value > global_p95
df_trace_window['is_breach'] = df_trace_window['value'] > df_trace_window['global_p95']

# aggregate per (cmdb_id, trace_name) for pairs that have at least one breach
breach_rows = df_trace_window[df_trace_window['is_breach']].copy()

if breach_rows.shape[0] == 0:
    trace_breach_top = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_of_breaches','earliest_breach_timestamp',
        'max_value_in_window','global_p95','severity'
    ])
else:
    agg = (
        breach_rows
        .groupby(['cmdb_id', 'trace_name'])
        .agg(
            count_of_breaches=('is_breach', 'sum'),
            earliest_breach_timestamp=('ts', 'min')
        )
        .reset_index()
    )

    max_window = (
        df_trace_window
        .groupby(['cmdb_id', 'trace_name'])['value']
        .max()
        .reset_index(name='max_value_in_window')
    )

    # join aggregates and global_p95 (global_p95 already per trace_name)
    trace_breach = (
        agg
        .merge(max_window, on=['cmdb_id', 'trace_name'], how='left')
        .merge(global_p95_trace, on='trace_name', how='left')
    )

    # compute severity, handle zero global_p95
    trace_breach['severity'] = np.where(
        trace_breach['global_p95'] > 0,
        (trace_breach['max_value_in_window'] - trace_breach['global_p95']) / trace_breach['global_p95'],
        np.nan
    )

    # sort by severity desc and keep top 20
    trace_breach_top = trace_breach.sort_values('severity', ascending=False)[
        ['cmdb_id','trace_name','count_of_breaches','earliest_breach_timestamp',
         'max_value_in_window','global_p95','severity']
    ].head(20).reset_index(drop=True)

    # round numeric columns for compactness
    trace_breach_top['max_value_in_window'] = trace_breach_top['max_value_in_window'].round(6)
    trace_breach_top['global_p95'] = trace_breach_top['global_p95'].round(6)
    trace_breach_top['severity'] = trace_breach_top['severity'].round(6)

# Display the compact table (up to 20 rows)
trace_breach_top

```
Out[5]:
```
```python
# Build a concise plain-English summary of the trace breach results (reuse variables from kernel if present)
import pandas as pd

try:
    tb = trace_breach_top.copy()
except NameError:
    # fallback: load and compute minimal table if not present
    import numpy as np, pytz
    tz = pytz.timezone('Asia/Shanghai')
    df_trace = pd.read_csv('trace.csv')
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
    global_p95_trace = df_trace.groupby('trace_name')['value'].agg(global_p95=lambda x: x.quantile(0.95)).reset_index()
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
    end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)
    df_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].merge(global_p95_trace, on='trace_name', how='left')
    df_window['is_breach'] = df_window['value'] > df_window['global_p95']
    breach_rows = df_window[df_window['is_breach']].copy()
    if breach_rows.shape[0] == 0:
        tb = pd.DataFrame(columns=['cmdb_id','trace_name','count_of_breaches','earliest_breach_timestamp','max_value_in_window','global_p95','severity'])
    else:
        agg = breach_rows.groupby(['cmdb_id','trace_name']).agg(count_of_breaches=('is_breach','sum'), earliest_breach_timestamp=('ts','min')).reset_index()
        max_window = df_window.groupby(['cmdb_id','trace_name'])['value'].max().reset_index(name='max_value_in_window')
        tb = agg.merge(max_window, on=['cmdb_id','trace_name']).merge(global_p95_trace, on='trace_name', how='left')
        tb['severity'] = ((tb['max_value_in_window'] - tb['global_p95']) / tb['global_p95']).replace([np.inf, -np.inf], np.nan)
        tb = tb.sort_values('severity', ascending=False).head(20)
        tb['max_value_in_window'] = tb['max_value_in_window'].round(6)
        tb['global_p95'] = tb['global_p95'].round(6)
        tb['severity'] = tb['severity'].round(6)

# Compose plain-English summary
total_pairs = int(tb.shape[0])
lines = []
lines.append("Incident window: 2020-05-31 02:30:00+08 to 2020-05-31 03:00:00+08.")
lines.append(f"Number of (cmdb_id, trace_name) pairs that exceeded their global P95: {total_pairs}.")
lines.append("Top breaches by severity (up to 5):")

for _, r in tb.head(5).iterrows():
    ts = r['earliest_breach_timestamp']
    ts_str = pd.to_datetime(ts).tz_convert('Asia/Shanghai').isoformat() if pd.notnull(ts) else "N/A"
    lines.append(
        f"- {r['cmdb_id']} / {r['trace_name']}: breaches={int(r['count_of_breaches'])}, "
        f"earliest={ts_str}, max_in_window={float(r['max_value_in_window'])}, "
        f"global_p95={float(r['global_p95'])}, severity={float(r['severity']):.3f}"
    )

lines.append("")
lines.append("Observation: The highest-severity anomalies are on os_021 (trace.self.csf.* durations) and several docker containers showing elevated JDBC and remoteprocess durations. This points to application-level latency concentrated on specific containers and the OS node during the incident window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                        trace_name  count_of_breaches earliest_breach_timestamp  max_value_in_window  global_p95  severity
0       os_021                      trace.self.csf.duration_mean                  4 2020-05-31 02:48:00+08:00             4.905000    0.669450  6.326912
1   docker_005                      trace.self.jdbc.duration_p95                  2 2020-05-31 02:30:00+08:00             0.010000    0.003000  2.333333
2   docker_007                      trace.self.jdbc.duration_p95                  1 2020-05-31 02:33:00+08:00             0.010000    0.003000  2.333333
3   docker_006                      trace.self.jdbc.duration_p95                  3 2020-05-31 02:30:00+08:00             0.010000    0.003000  2.333333
4   docker_008                      trace.self.jdbc.duration_p95                  1 2020-05-31 02:33:00+08:00             0.009000    0.003000  2.000000
5       os_021                       trace.self.csf.duration_p95                  5 2020-05-31 02:48:00+08:00             6.185000    2.078052  1.976344
6   docker_004    trace.to_docker_006.remoteprocess.duration_p95                  1 2020-05-31 02:33:00+08:00             0.065300    0.045025  0.450305
7   docker_004                trace.self.flyremote.duration_mean                  1 2020-05-31 02:50:00+08:00             0.016056    0.011537  0.391648
8   docker_006  trace.from_docker_004.remoteprocess.duration_p95                  1 2020-05-31 02:33:00+08:00             0.065300    0.048178  0.355404
9   docker_007  trace.from_docker_001.remoteprocess.duration_p95                  2 2020-05-31 02:33:00+08:00             0.058450    0.043302  0.349807
10  docker_003    trace.to_docker_006.remoteprocess.duration_p95                  2 2020-05-31 02:33:00+08:00             0.060750    0.045025  0.349250
11  docker_002                      trace.self.jdbc.duration_p95                  2 2020-05-31 02:39:00+08:00             0.004000    0.003000  0.333333
12  docker_001                      trace.self.jdbc.duration_p95                  1 2020-05-31 02:39:00+08:00             0.004000    0.003000  0.333333
13  docker_007                     trace.self.jdbc.duration_mean                  1 2020-05-31 02:33:00+08:00             0.004039    0.003127  0.291674
14  docker_005  trace.from_docker_004.remoteprocess.duration_p95                  2 2020-05-31 02:33:00+08:00             0.062000    0.048178  0.286908
15  docker_002    trace.to_docker_008.remoteprocess.duration_p95                  2 2020-05-31 02:33:00+08:00             0.055200    0.043067  0.281709
16  docker_002                trace.self.flyremote.duration_mean                  5 2020-05-31 02:31:00+08:00             0.014685    0.011537  0.272817
17  docker_007  trace.from_docker_002.remoteprocess.duration_p95                  2 2020-05-31 02:33:00+08:00             0.057000    0.044832  0.271399
18  docker_002                     trace.self.jdbc.duration_mean                  6 2020-05-31 02:40:00+08:00             0.003885    0.003127  0.242425
19  docker_001                     trace.self.jdbc.duration_mean                  2 2020-05-31 02:39:00+08:00             0.003872    0.003127  0.238267```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# reuse timezone and data if available
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure numeric value and timezone-aware timestamp column (Asia/Shanghai)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz
    df_metric['ts'] = pd.to_datetime(df_metric['ts'], utc=True).dt.tz_convert(tz)

# compute or reuse global P95 per (cmdb_id, kpi_name) BEFORE filtering
try:
    metric_global_p95
except NameError:
    metric_global_p95 = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# define target KPI list
targets = [
    ('db_011', 'LFParaWrite_Per_Sec'),
    ('db_011', 'LFSync_Per_Sec'),
    ('os_018', 'Disk_await')
]

# incident window bounds (inclusive start, exclusive end) in Asia/Shanghai
start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)

# filter rows to window and targets
mask_targets = pd.Series(False, index=df_metric.index)
for cmdb, kpi in targets:
    mask_targets |= ((df_metric['cmdb_id'] == cmdb) & (df_metric['kpi_name'] == kpi))

df_selected = df_metric[
    mask_targets & (df_metric['ts'] >= start) & (df_metric['ts'] < end)
].copy()

# prepare compact table sorted by timestamp ascending and limited to 50 rows
df_selected_table = (
    df_selected
    .sort_values('ts', ascending=True)
    .loc[:, ['ts', 'cmdb_id', 'kpi_name', 'value']]
    .head(50)
    .reset_index(drop=True)
)
# rename ts -> timestamp for requested column name
df_selected_table = df_selected_table.rename(columns={'ts': 'timestamp'})

# For each target KPI, find earliest breach within the incident window where value > global_p95
# Build a DataFrame with one row per target
breach_rows_list = []
for cmdb, kpi in targets:
    # get global p95 for this pair
    gp = metric_global_p95[
        (metric_global_p95['cmdb_id'] == cmdb) & (metric_global_p95['kpi_name'] == kpi)
    ]
    global_p95_val = gp['global_p95'].iloc[0] if not gp.empty else np.nan

    # filter selected window rows for this pair
    df_pair = df_selected[
        (df_selected['cmdb_id'] == cmdb) & (df_selected['kpi_name'] == kpi)
    ].copy()

    # find breaches (value > global_p95), earliest breach timestamp and breach value
    if df_pair.shape[0] == 0 or pd.isna(global_p95_val):
        earliest_ts = pd.NaT
        breach_val = np.nan
    else:
        df_breach = df_pair[df_pair['value'] > global_p95_val].sort_values('ts', ascending=True)
        if df_breach.shape[0] == 0:
            earliest_ts = pd.NaT
            breach_val = np.nan
        else:
            earliest_ts = df_breach['ts'].iloc[0]
            breach_val = df_breach['value'].iloc[0]

    breach_rows_list.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': global_p95_val,
        'earliest_breach_timestamp': earliest_ts,
        'breach_value': breach_val
    })

df_earliest_breaches = pd.DataFrame(breach_rows_list)

# Round numeric columns for compactness
df_earliest_breaches['global_p95'] = df_earliest_breaches['global_p95'].round(6)
df_earliest_breaches['breach_value'] = df_earliest_breaches['breach_value'].round(6)

# Ensure timestamp column is tz-aware and displayed; keep compact output
# Final outputs: df_selected_table (max 50 rows), df_earliest_breaches (3 rows)
df_selected_table, df_earliest_breaches

```
Out[6]:
```
```python
# Build a concise plain-English summary using prior results (df_selected_table and df_earliest_breaches) if available.
import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

try:
    sel = df_selected_table.copy()
    eb = df_earliest_breaches.copy()
except NameError:
    # fallback: reload and recompute minimal required pieces
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_global_p95 = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(global_p95=lambda x: x.quantile(0.95)).reset_index()
    targets = [
        ('db_011', 'LFParaWrite_Per_Sec'),
        ('db_011', 'LFSync_Per_Sec'),
        ('os_018', 'Disk_await')
    ]
    start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
    end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)
    mask = pd.Series(False, index=df_metric.index)
    for cmdb,kpi in targets:
        mask |= ((df_metric['cmdb_id']==cmdb) & (df_metric['kpi_name']==kpi))
    sel = df_metric[mask & (df_metric['ts']>=start) & (df_metric['ts']<end)].sort_values('ts').loc[:, ['ts','cmdb_id','kpi_name','value']].head(50)
    sel = sel.rename(columns={'ts':'timestamp'})
    rows = []
    for cmdb,kpi in targets:
        gp = metric_global_p95[(metric_global_p95['cmdb_id']==cmdb)&(metric_global_p95['kpi_name']==kpi)]
        gpv = gp['global_p95'].iloc[0] if not gp.empty else np.nan
        df_pair = sel[(sel['cmdb_id']==cmdb)&(sel['kpi_name']==kpi)].copy()
        if df_pair.empty or pd.isna(gpv):
            earliest = pd.NaT; val = np.nan
        else:
            breach = df_pair[df_pair['value']>gpv].sort_values('timestamp')
            if breach.empty:
                earliest = pd.NaT; val = np.nan
            else:
                earliest = breach['timestamp'].iloc[0]; val = breach['value'].iloc[0]
        rows.append({'cmdb_id':cmdb,'kpi_name':kpi,'global_p95':round(float(gpv),6) if not pd.isna(gpv) else np.nan,
                     'earliest_breach_timestamp':earliest,'breach_value': (round(float(val),6) if not pd.isna(val) else np.nan)})
    eb = pd.DataFrame(rows)

# Compose summary string
n_rows = int(sel.shape[0])
lines = []
lines.append(f"Selected rows in incident window (sorted by timestamp, limited to 50 shown): {n_rows} rows (columns: timestamp, cmdb_id, kpi_name, value).")
lines.append("Displayed series: db_011/LFParaWrite_Per_Sec, db_011/LFSync_Per_Sec, os_018/Disk_await.")
lines.append("")
lines.append("Earliest breach per KPI (value > global P95):")
for _, r in eb.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    gp = r['global_p95']
    ts = r['earliest_breach_timestamp']
    ts_str = pd.NaT if pd.isna(ts) else pd.to_datetime(ts).tz_convert(tz).isoformat()
    val = r['breach_value']
    if pd.isna(ts):
        lines.append(f"- {cmdb} / {kpi}: no breach in window (global_p95={gp}).")
    else:
        lines.append(f"- {cmdb} / {kpi}: earliest_breach_timestamp={ts_str}, breach_value={val}, global_p95={gp}.")

lines.append("")
lines.append("Quick observation: LFParaWrite_Per_Sec on db_011 had a dramatic breach (value 17.89 vs P95 0.14) at 2020-05-31T02:51:00+08:00; LFSync_Per_Sec on db_011 first breached at 02:37 with value 0.47 vs P95 0.451; os_018 Disk_await first breached at 02:42 with value 20.29 vs P95 0.638.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp cmdb_id             kpi_name  value
0  2020-05-31 02:30:00+08:00  db_011  LFParaWrite_Per_Sec   0.06
1  2020-05-31 02:30:00+08:00  db_011       LFSync_Per_Sec   0.40
2  2020-05-31 02:30:00+08:00  os_018           Disk_await   0.00
3  2020-05-31 02:31:00+08:00  db_011  LFParaWrite_Per_Sec   0.06
4  2020-05-31 02:31:00+08:00  db_011       LFSync_Per_Sec   0.38
..                       ...     ...                  ...    ...
45 2020-05-31 02:45:00+08:00  db_011  LFParaWrite_Per_Sec   0.05
46 2020-05-31 02:45:00+08:00  db_011       LFSync_Per_Sec   0.43
47 2020-05-31 02:45:00+08:00  os_018           Disk_await   0.00
48 2020-05-31 02:46:00+08:00  db_011  LFParaWrite_Per_Sec   0.05
49 2020-05-31 02:46:00+08:00  db_011       LFSync_Per_Sec   0.38

[50 rows x 4 columns],   cmdb_id             kpi_name  global_p95 earliest_breach_timestamp  breach_value
0  db_011  LFParaWrite_Per_Sec       0.140 2020-05-31 02:51:00+08:00         17.89
1  db_011       LFSync_Per_Sec       0.451 2020-05-31 02:37:00+08:00          0.47
2  os_018           Disk_await       0.638 2020-05-31 02:42:00+08:00         20.29)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# reuse existing df_metric and tz if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure value numeric and ts timezone-aware in Asia/Shanghai
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # normalize ts to tz-aware Asia/Shanghai
    df_metric['ts'] = pd.to_datetime(df_metric['ts'], utc=True).dt.tz_convert(tz)

# target exact timestamps (Asia/Shanghai)
ts_list = [
    pd.Timestamp('2020-05-31 02:37:00').tz_localize(tz),
    pd.Timestamp('2020-05-31 02:51:00').tz_localize(tz)
]

# filter rows where timestamp equals either target minute
mask_time = df_metric['ts'].isin(ts_list)

# filter for os_* Disk_await OR db_011 specific KPIs
mask_os_disk = df_metric['cmdb_id'].str.startswith('os_') & (df_metric['kpi_name'] == 'Disk_await')
mask_db011 = (df_metric['cmdb_id'] == 'db_011') & (df_metric['kpi_name'].isin(['LFSync_Per_Sec', 'LFParaWrite_Per_Sec']))

df_filtered = df_metric[mask_time & (mask_os_disk | mask_db011)].copy()

# prepare output table with requested columns, sorted by timestamp asc then value desc, limited to 50 rows
combined_table = (
    df_filtered
    .loc[:, ['ts', 'cmdb_id', 'kpi_name', 'value']]
    .rename(columns={'ts': 'timestamp'})
    .sort_values(['timestamp', 'value'], ascending=[True, False])
    .head(50)
    .reset_index(drop=True)
)

# For each timestamp, identify top os_* node by Disk_await value
os_rows = df_filtered[mask_os_disk].copy()
top_os_list = []
for ts in ts_list:
    rows_at_ts = os_rows[os_rows['ts'] == ts]
    if rows_at_ts.shape[0] == 0:
        top_os_list.append({'timestamp': ts, 'top_os_node': None, 'value': np.nan})
    else:
        idx = rows_at_ts['value'].idxmax()
        top = rows_at_ts.loc[idx]
        top_os_list.append({'timestamp': ts, 'top_os_node': top['cmdb_id'], 'value': top['value']})

top_os_summary = pd.DataFrame(top_os_list)
# display timestamps in same tz and keep columns compact
top_os_summary['timestamp'] = pd.to_datetime(top_os_summary['timestamp']).dt.tz_convert(tz)

# Final compact outputs
combined_table, top_os_summary

```
Out[7]:
```
```python
# Build a concise plain-English summary of the extracted rows and top OS nodes (reuse kernel variables if present)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    combined_table
    top_os_summary
except NameError:
    # Fallback: recompute minimal selection if variables are not present
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    ts_list = [pd.Timestamp('2020-05-31 02:37:00').tz_localize(tz), pd.Timestamp('2020-05-31 02:51:00').tz_localize(tz)]
    mask_time = df_metric['ts'].isin(ts_list)
    mask_os_disk = df_metric['cmdb_id'].str.startswith('os_') & (df_metric['kpi_name'] == 'Disk_await')
    mask_db011 = (df_metric['cmdb_id'] == 'db_011') & (df_metric['kpi_name'].isin(['LFSync_Per_Sec', 'LFParaWrite_Per_Sec']))
    df_filtered = df_metric[mask_time & (mask_os_disk | mask_db011)].copy()
    combined_table = df_filtered.loc[:, ['ts','cmdb_id','kpi_name','value']].rename(columns={'ts':'timestamp'}).sort_values(['timestamp','value'], ascending=[True, False]).head(50).reset_index(drop=True)
    os_rows = df_filtered[mask_os_disk].copy()
    top_os_list = []
    for ts in ts_list:
        rows_at_ts = os_rows[os_rows['ts'] == ts]
        if rows_at_ts.shape[0] == 0:
            top_os_list.append({'timestamp': ts, 'top_os_node': None, 'value': float('nan')})
        else:
            idx = rows_at_ts['value'].idxmax()
            top = rows_at_ts.loc[idx]
            top_os_list.append({'timestamp': ts, 'top_os_node': top['cmdb_id'], 'value': top['value']})
    top_os_summary = pd.DataFrame(top_os_list)
    top_os_summary['timestamp'] = pd.to_datetime(top_os_summary['timestamp']).dt.tz_convert(tz)

# Compose the plain-English summary string
n_rows = int(combined_table.shape[0])
lines = []
lines.append(f"Returned table: {n_rows} rows (columns: timestamp, cmdb_id, kpi_name, value), sorted by timestamp then value desc. Contains os_* Disk_await rows and db_011 LFSync/LFParaWrite rows at the two requested minutes.")
lines.append("")
lines.append("Top os_* nodes by Disk_await at each timestamp:")
for _, r in top_os_summary.iterrows():
    ts_iso = pd.to_datetime(r['timestamp']).tz_convert(tz).isoformat()
    node = r['top_os_node'] if pd.notnull(r['top_os_node']) else "None"
    val = r['value']
    lines.append(f"- {ts_iso}: {node}, value={val}")

lines.append("")
lines.append("Quick observation: At 2020-05-31 02:37+08 the highest Disk_await was os_012 (4.55 ms). At 2020-05-31 02:51+08 the highest Disk_await was os_005 (2.44 ms).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp cmdb_id    kpi_name  value
0  2020-05-31 02:37:00+08:00  os_012  Disk_await   4.55
1  2020-05-31 02:37:00+08:00  os_016  Disk_await   3.50
2  2020-05-31 02:37:00+08:00  os_014  Disk_await   1.33
3  2020-05-31 02:37:00+08:00  os_003  Disk_await   1.06
4  2020-05-31 02:37:00+08:00  os_004  Disk_await   0.85
..                       ...     ...         ...    ...
43 2020-05-31 02:51:00+08:00  os_018  Disk_await   0.00
44 2020-05-31 02:51:00+08:00  os_019  Disk_await   0.00
45 2020-05-31 02:51:00+08:00  os_020  Disk_await   0.00
46 2020-05-31 02:51:00+08:00  os_021  Disk_await   0.00
47 2020-05-31 02:51:00+08:00  os_022  Disk_await   0.00

[48 rows x 4 columns],                   timestamp top_os_node  value
0 2020-05-31 02:37:00+08:00      os_012   4.55
1 2020-05-31 02:51:00+08:00      os_005   2.44)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse stateful variables if present, otherwise load/compute
tz = pytz.timezone('Asia/Shanghai')

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure numeric and timezone-aware timestamp in Asia/Shanghai
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure global P95 per (cmdb_id, kpi_name) computed using the entire file (before any filtering)
try:
    metric_global_p95
except NameError:
    metric_global_p95 = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# Incident window (inclusive start, exclusive end) in Asia/Shanghai
start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)

# Filter metric rows to the window and to cmdb_id starting with 'db_'
mask_window = (df_metric['ts'] >= start) & (df_metric['ts'] < end)
mask_db = df_metric['cmdb_id'].str.startswith('db_')
df_window_db = df_metric[mask_window & mask_db].copy()

# Merge global_p95 into windowed rows
df_window_db = df_window_db.merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Identify breach rows (value > global_p95)
df_window_db['is_breach'] = df_window_db['value'] > df_window_db['global_p95']

# Aggregate per (cmdb_id, kpi_name): count_of_breaches, earliest_breach_timestamp, max_value_in_window, global_p95, severity_per_kpi
def earliest_breach_ts(series_ts, series_is_breach):
    b = series_ts[series_is_breach]
    return b.min() if not b.empty else pd.NaT

per_kpi = (
    df_window_db
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_of_breaches=('is_breach', 'sum'),
        earliest_breach_timestamp=('ts', lambda x: earliest_breach_ts(x, df_window_db.loc[x.index, 'is_breach'])),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
)

# compute severity per KPI safely
per_kpi['severity_per_kpi'] = np.where(
    per_kpi['global_p95'] > 0,
    (per_kpi['max_value_in_window'] - per_kpi['global_p95']) / per_kpi['global_p95'],
    np.nan
)

# Now aggregate per cmdb_id
def min_valid_timestamp(series):
    # return min of non-na timestamps or NaT
    s = pd.to_datetime(series.dropna())
    return s.min() if not s.empty else pd.NaT

per_db = (
    per_kpi
    .groupby('cmdb_id', as_index=False)
    .agg(
        breached=('count_of_breaches', lambda x: (x > 0).any()),
        earliest_breach_timestamp=('earliest_breach_timestamp', lambda x: min_valid_timestamp(x)),
        max_severity=('severity_per_kpi', 'max'),
        number_of_distinct_kpis_that_breached=('count_of_breaches', lambda x: int((x > 0).sum()))
    )
)

# Sort by max_severity desc and keep top 20
per_db_sorted = per_db.sort_values('max_severity', ascending=False).head(20).reset_index(drop=True)

# Round numeric fields and keep timestamps timezone-aware
per_db_sorted['max_severity'] = per_db_sorted['max_severity'].round(6)
# Ensure earliest_breach_timestamp is tz-aware Asia/Shanghai or NaT
per_db_sorted['earliest_breach_timestamp'] = pd.to_datetime(per_db_sorted['earliest_breach_timestamp']).dt.tz_convert(tz)

# Count total number of db_* components that had breached == True (over the whole set, not just top 20)
total_breached_dbs = int(per_db['breached'].sum())

# Final compact outputs
per_db_sorted, total_breached_dbs

```
Out[8]:
```
```python
# Build a concise plain-English summary of the DB-level breach aggregation (reuse kernel variables if present)
import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

try:
    top_db = per_db_sorted.copy()
    total_breached = int(total_breached_dbs)
except NameError:
    # fallback: recompute minimal aggregates if variables are missing
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_global_p95 = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(global_p95=lambda x: x.quantile(0.95)).reset_index()
    start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
    end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)
    df_window_db = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end) & df_metric['cmdb_id'].str.startswith('db_')].merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')
    df_window_db['is_breach'] = df_window_db['value'] > df_window_db['global_p95']
    per_kpi = df_window_db.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        count_of_breaches=('is_breach','sum'),
        earliest_breach_timestamp=('ts', lambda x: x[df_window_db.loc[x.index,'is_breach']].min() if any(df_window_db.loc[x.index,'is_breach']) else pd.NaT),
        max_value_in_window=('value','max'),
        global_p95=('global_p95','first')
    )
    per_kpi['severity_per_kpi'] = ((per_kpi['max_value_in_window'] - per_kpi['global_p95']) / per_kpi['global_p95']).replace([pd.NA, pd.NaT], pd.NA)
    def min_valid_timestamp(s): 
        s2 = pd.to_datetime(s.dropna())
        return s2.min() if not s2.empty else pd.NaT
    per_db = per_kpi.groupby('cmdb_id', as_index=False).agg(
        breached=('count_of_breaches', lambda x: (x>0).any()),
        earliest_breach_timestamp=('earliest_breach_timestamp', lambda x: min_valid_timestamp(x)),
        max_severity=('severity_per_kpi', 'max'),
        number_of_distinct_kpis_that_breached=('count_of_breaches', lambda x: int((x>0).sum()))
    )
    top_db = per_db.sort_values('max_severity', ascending=False).head(20).reset_index(drop=True)
    top_db['max_severity'] = top_db['max_severity'].round(6)
    top_db['earliest_breach_timestamp'] = pd.to_datetime(top_db['earliest_breach_timestamp']).dt.tz_convert(tz)
    total_breached = int(per_db['breached'].sum())

# Compose summary string
lines = []
lines.append("Incident window: 2020-05-31 02:30:00+08 to 2020-05-31 03:00:00+08.")
lines.append(f"Total number of db_* components that breached at least one KPI: {total_breached}.")
lines.append("Top DB components by max_severity (up to 5):")

for _, r in top_db.head(5).iterrows():
    cid = r['cmdb_id']
    breached = bool(r['breached'])
    ts = r['earliest_breach_timestamp']
    ts_str = pd.NaT if pd.isna(ts) else pd.to_datetime(ts).tz_convert(tz).isoformat()
    max_sev = float(r['max_severity']) if pd.notna(r['max_severity']) else float('nan')
    num_kpis = int(r['number_of_distinct_kpis_that_breached'])
    lines.append(f"- {cid}: breached={breached}, earliest_breach={ts_str}, max_severity={max_sev:.6f}, distinct_kpis_breached={num_kpis}")

lines.append("")
lines.append("Quick observation: db_011 stands out with the highest severity (>>100x) and many distinct KPIs breached. Several other DB nodes (db_002, db_009, db_012, db_003, ...) also show substantial relative excursions above their historical P95s during the incident window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  breached earliest_breach_timestamp  max_severity  number_of_distinct_kpis_that_breached
0   db_011      True 2020-05-31 02:35:00+08:00    126.785714                                     18
1   db_002      True 2020-05-31 02:31:00+08:00     81.439024                                     17
2   db_009      True 2020-05-31 02:38:00+08:00     17.117954                                     15
3   db_012      True 2020-05-31 02:30:00+08:00     12.420366                                     17
4   db_003      True 2020-05-31 02:31:00+08:00      8.181243                                     20
5   db_013      True 2020-05-31 02:30:00+08:00      6.517730                                     14
6   db_005      True 2020-05-31 02:31:00+08:00      2.094496                                     19
7   db_004      True 2020-05-31 02:30:00+08:00      1.586275                                     21
8   db_010      True 2020-05-31 02:33:00+08:00      0.553398                                     14
9   db_001      True 2020-05-31 02:31:00+08:00      0.500000                                     15
10  db_006      True 2020-05-31 02:30:00+08:00      0.500000                                     16
11  db_007      True 2020-05-31 02:33:00+08:00      0.463415                                     19
12  db_008      True 2020-05-31 02:32:00+08:00      0.250000                                     15, 13)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables where possible
tz = pytz.timezone('Asia/Shanghai')

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure numeric value and timezone-aware timestamp column in Asia/Shanghai
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    df_metric['ts'] = pd.to_datetime(df_metric['ts'], utc=True).dt.tz_convert(tz)

# ensure global P95 per (cmdb_id, kpi_name) computed using the entire file (before filtering)
try:
    metric_global_p95
except NameError:
    metric_global_p95 = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# incident window bounds (inclusive start, exclusive end) in Asia/Shanghai
start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)

# filter to window and to db_011
df_window_db011 = df_metric[
    (df_metric['ts'] >= start) & (df_metric['ts'] < end) & (df_metric['cmdb_id'] == 'db_011')
].copy()

# merge global_p95 for db_011 KPIs
df_window_db011 = df_window_db011.merge(
    metric_global_p95[metric_global_p95['cmdb_id'] == 'db_011'],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# detect breaches within window
df_window_db011['is_breach'] = df_window_db011['value'] > df_window_db011['global_p95']

# aggregate per kpi_name
agg = (
    df_window_db011
    .groupby('kpi_name', as_index=False)
    .agg(
        count_of_breaches=('is_breach', 'sum'),
        earliest_breach_timestamp=('ts', lambda s: s[df_window_db011.loc[s.index, 'is_breach']].min() if s.size>0 else pd.NaT),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
)

# compute severity safely
agg['severity'] = np.where(
    agg['global_p95'] > 0,
    (agg['max_value_in_window'] - agg['global_p95']) / agg['global_p95'],
    np.nan
)

# sort by earliest_breach_timestamp asc (NaT last) then severity desc
agg_sorted = agg.sort_values(
    ['earliest_breach_timestamp', 'severity'],
    ascending=[True, False],
    na_position='last'
).reset_index(drop=True)

# round numeric columns for compactness and limit to 50 rows
agg_sorted['max_value_in_window'] = agg_sorted['max_value_in_window'].round(6)
agg_sorted['global_p95'] = agg_sorted['global_p95'].round(6)
agg_sorted['severity'] = agg_sorted['severity'].round(6)

db011_kpi_summary = agg_sorted[['kpi_name', 'count_of_breaches', 'earliest_breach_timestamp',
                                'max_value_in_window', 'global_p95', 'severity']].head(50)

db011_kpi_summary

```
Out[9]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing db011_kpi_summary if present, otherwise load minimal
try:
    df = db011_kpi_summary.copy()
except NameError:
    # fallback: recompute (minimal) if variable not in kernel
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_global_p95 = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(global_p95=lambda x: x.quantile(0.95)).reset_index()
    start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
    end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)
    df_window_db011 = df_metric[(df_metric['ts']>=start)&(df_metric['ts']<end)&(df_metric['cmdb_id']=='db_011')].merge(metric_global_p95[metric_global_p95['cmdb_id']=='db_011'], on=['cmdb_id','kpi_name'], how='left')
    df_window_db011['is_breach'] = df_window_db011['value'] > df_window_db011['global_p95']
    agg = (
        df_window_db011.groupby('kpi_name', as_index=False)
        .agg(
            count_of_breaches=('is_breach','sum'),
            earliest_breach_timestamp=('ts', lambda s: s[df_window_db011.loc[s.index,'is_breach']].min() if s.size>0 else pd.NaT),
            max_value_in_window=('value','max'),
            global_p95=('global_p95','first')
        )
    )
    agg['severity'] = ((agg['max_value_in_window'] - agg['global_p95']) / agg['global_p95']).round(6)
    df = agg[['kpi_name','count_of_breaches','earliest_breach_timestamp','max_value_in_window','global_p95','severity']]

# Ensure timestamps are tz-aware Asia/Shanghai
df['earliest_breach_timestamp'] = pd.to_datetime(df['earliest_breach_timestamp']).dt.tz_convert(tz)

# Summary stats
total_kpis = int(df.shape[0])
breached_kpis = df[df['count_of_breaches']>0].copy()
num_breached = int(breached_kpis.shape[0])

# earliest overall breach for db_011 (if any)
if not breached_kpis.empty:
    overall_earliest_row = breached_kpis.sort_values('earliest_breach_timestamp', na_position='last').iloc[0]
    overall_earliest_ts = pd.to_datetime(overall_earliest_row['earliest_breach_timestamp']).tz_convert(tz).isoformat()
else:
    overall_earliest_ts = "None"

# top 5 KPIs by severity (exclude NaN severities)
top_sev = breached_kpis.dropna(subset=['severity']).sort_values('severity', ascending=False).head(5)

lines = []
lines.append("db_011 KPI breach summary for 2020-05-31 02:30:00+08 through 03:00:00+08:")
lines.append(f"- Total db_011 KPIs analyzed: {total_kpis}")
lines.append(f"- Number of KPIs that breached their global P95 in the window: {num_breached}")
lines.append(f"- Earliest breach across db_011 KPIs: {overall_earliest_ts}")
lines.append("")
lines.append("Top KPIs by severity (up to 5):")
if top_sev.empty:
    lines.append("- (no KPI with a defined severity breach found)")
else:
    for _, r in top_sev.iterrows():
        k = r['kpi_name']
        cnt = int(r['count_of_breaches'])
        ets = pd.to_datetime(r['earliest_breach_timestamp']).tz_convert(tz).isoformat() if pd.notnull(r['earliest_breach_timestamp']) else "None"
        mv = float(r['max_value_in_window']) if pd.notnull(r['max_value_in_window']) else float('nan')
        gp = float(r['global_p95']) if pd.notnull(r['global_p95']) else float('nan')
        sev = float(r['severity']) if pd.notnull(r['severity']) else float('nan')
        lines.append(f"- {k}: breaches={cnt}, earliest={ets}, max_in_window={mv}, global_p95={gp}, severity={sev:.6f}")

lines.append("")
lines.append("Notable observation: Several db_011 KPIs show very large relative excursions (high severity). In particular, LFSync_Per_Sec and LFParaWrite_Per_Sec are among the highest-severity breaches (large multiples above their historical P95).")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  count_of_breaches earliest_breach_timestamp  max_value_in_window  global_p95   severity
0          CPU_free_pct                  2 2020-05-31 02:35:00+08:00            99.062766   99.050633   0.000122
1        LFSync_Per_Sec                  2 2020-05-31 02:37:00+08:00            37.990000    0.451000  83.235033
2           Session_pct                  2 2020-05-31 02:38:00+08:00             0.066000    0.060300   0.094527
3                DbTime                  2 2020-05-31 02:38:00+08:00            10.800000   10.320000   0.046512
4   DFParaWrite_Per_Sec                  1 2020-05-31 02:42:00+08:00             0.180000    0.085000   1.117647
..                  ...                ...                       ...                  ...         ...        ...
40      SctRead_Per_Sec                  0                       NaT             0.000000    0.000000        NaN
41       Sess_Used_Temp                  0                       NaT             0.000000    0.000000        NaN
42       Sess_Used_Undo                  0                       NaT             0.000000    0.000000        NaN
43          TPS_Per_Sec                  0                       NaT             0.000000    0.000000        NaN
44          TempTbs_Pct                  0                       NaT             0.000000    0.000000        NaN

[45 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
from itertools import product

# reuse existing stateful variables if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    metric_global_p95
except NameError:
    # compute global p95 per (cmdb_id, kpi_name) using entire file
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    metric_global_p95 = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# ensure numeric and timezone-aware timestamp column (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define incident window bounds (inclusive start, exclusive end)
start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)

# target KPIs
target_kpis = ['LFSync_Per_Sec', 'LFParaWrite_Per_Sec']

# get all db_* cmdb_ids from the global p95 table (ensures we consider nodes even if no window data)
db_ids = sorted(metric_global_p95[metric_global_p95['cmdb_id'].str.startswith('db_')]['cmdb_id'].unique())

# build a DataFrame of all (db_id, target_kpi) combinations
pairs = pd.DataFrame(list(product(db_ids, target_kpis)), columns=['cmdb_id', 'kpi_name'])

# filter metric rows to window and relevant KPIs and dbs
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end) & (df_metric['cmdb_id'].str.startswith('db_')) & (df_metric['kpi_name'].isin(target_kpis))].copy()

# merge global_p95 into window rows for breach detection
df_window = df_window.merge(metric_global_p95[['cmdb_id', 'kpi_name', 'global_p95']],
                            on=['cmdb_id', 'kpi_name'],
                            how='left')

# per-pair aggregates from window
if df_window.shape[0] > 0:
    per_pair_window = (
        df_window
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            max_value_in_window=('value', 'max'),
            # count of values > global_p95 within window
            count_of_breaches=('value', lambda s: int((s > df_window.loc[s.index, 'global_p95']).sum())),
            # earliest breach timestamp
            earliest_breach_timestamp=('ts', lambda s: (s[df_window.loc[s.index, 'value'] > df_window.loc[s.index, 'global_p95']].min()) if any(df_window.loc[s.index, 'value'] > df_window.loc[s.index, 'global_p95']) else pd.NaT)
        )
    )
else:
    per_pair_window = pd.DataFrame(columns=['cmdb_id','kpi_name','max_value_in_window','count_of_breaches','earliest_breach_timestamp'])

# merge per-pair results with global_p95 (to include pairs with no window rows)
result = pairs.merge(metric_global_p95[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')
result = result.merge(per_pair_window, on=['cmdb_id','kpi_name'], how='left')

# fill NaNs for pairs with no data in window
result['max_value_in_window'] = result['max_value_in_window'].fillna(0)
result['count_of_breaches'] = result['count_of_breaches'].fillna(0).astype(int)
result['earliest_breach_timestamp'] = pd.to_datetime(result['earliest_breach_timestamp'])

# compute breached boolean and severity safely
result['breached'] = result['count_of_breaches'] > 0
result['severity'] = np.where(
    result['global_p95'] > 0,
    (result['max_value_in_window'] - result['global_p95']) / result['global_p95'],
    np.nan
)

# keep requested columns and sort by severity desc
db_io_spikes = result[
    ['cmdb_id', 'kpi_name', 'breached', 'earliest_breach_timestamp', 'max_value_in_window', 'global_p95', 'severity']
].sort_values(['severity'], ascending=[False]).head(50).reset_index(drop=True)

# rounding / formatting for compactness
db_io_spikes['max_value_in_window'] = db_io_spikes['max_value_in_window'].round(6)
db_io_spikes['global_p95'] = db_io_spikes['global_p95'].round(6)
db_io_spikes['severity'] = db_io_spikes['severity'].round(6)

# ensure earliest_breach_timestamp is timezone-aware Asia/Shanghai or NaT
db_io_spikes['earliest_breach_timestamp'] = pd.to_datetime(db_io_spikes['earliest_breach_timestamp']).dt.tz_localize(None)
db_io_spikes['earliest_breach_timestamp'] = pd.to_datetime(db_io_spikes['earliest_breach_timestamp']).dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')

# Display compact result (DataFrame variable)
db_io_spikes

```
Out[10]:
```
```python
# Summarize whether LFSync_Per_Sec and LFParaWrite_Per_Sec spikes are unique to db_011 or widespread.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    df = db_io_spikes.copy()
except NameError:
    # If the precomputed DataFrame is missing, load minimal recompute (fallback)
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_global_p95 = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(global_p95=lambda x: x.quantile(0.95)).reset_index()
    start = pd.Timestamp('2020-05-31 02:30:00').tz_localize(tz)
    end = pd.Timestamp('2020-05-31 03:00:00').tz_localize(tz)
    target_kpis = ['LFSync_Per_Sec', 'LFParaWrite_Per_Sec']
    db_ids = sorted(metric_global_p95[metric_global_p95['cmdb_id'].str.startswith('db_')]['cmdb_id'].unique())
    pairs = pd.DataFrame([(d,k) for d in db_ids for k in target_kpis], columns=['cmdb_id','kpi_name'])
    df_window = df_metric[(df_metric['ts']>=start)&(df_metric['ts']<end)&(df_metric['cmdb_id'].str.startswith('db_'))&(df_metric['kpi_name'].isin(target_kpis))].merge(metric_global_p95[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')
    per_pair = df_window.groupby(['cmdb_id','kpi_name'], as_index=False).agg(max_value_in_window=('value','max'), count_of_breaches=('value', lambda s: int((s>df_window.loc[s.index,'global_p95']).sum())), earliest_breach_timestamp=('ts', lambda s: (s[df_window.loc[s.index,'value']>df_window.loc[s.index,'global_p95']].min()) if any(df_window.loc[s.index,'value']>df_window.loc[s.index,'global_p95']) else pd.NaT))
    df = pairs.merge(metric_global_p95[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left').merge(per_pair, on=['cmdb_id','kpi_name'], how='left')
    df['max_value_in_window'] = df['max_value_in_window'].fillna(0)
    df['count_of_breaches'] = df['count_of_breaches'].fillna(0).astype(int)
    df['earliest_breach_timestamp'] = pd.to_datetime(df['earliest_breach_timestamp'])
    df['breached'] = df['count_of_breaches'] > 0
    df['severity'] = ((df['max_value_in_window'] - df['global_p95']) / df['global_p95']).replace([pd.NA, pd.NaT], pd.NA)

# Ensure types and formatting
df['earliest_breach_timestamp'] = pd.to_datetime(df['earliest_breach_timestamp']).dt.tz_convert(tz)
total_pairs = int(df.shape[0])
breached_pairs = df[df['breached']].copy()
num_breached = int(breached_pairs.shape[0])

# Top breached rows by severity
top_breached = breached_pairs.sort_values('severity', ascending=False).head(10)

lines = []
lines.append(f"Window: 2020-05-31 02:30+08 to 03:00+08. Checked LFSync_Per_Sec and LFParaWrite_Per_Sec for all db_* nodes ({total_pairs} pairs).")
lines.append(f"Number of (db, KPI) pairs that breached global P95: {num_breached}.")

lines.append("Top breaches by severity:")
for _, r in top_breached.iterrows():
    cm = r['cmdb_id']
    kpi = r['kpi_name']
    sev = float(r['severity']) if pd.notnull(r['severity']) else float('nan')
    maxv = float(r['max_value_in_window'])
    gp = float(r['global_p95']) if pd.notnull(r['global_p95']) else float('nan')
    ets = r['earliest_breach_timestamp']
    ets_str = pd.NaT if pd.isna(ets) else pd.to_datetime(ets).tz_convert(tz).isoformat()
    lines.append(f"- {cm} / {kpi}: severity={sev:.3f}, earliest={ets_str}, max={maxv}, p95={gp}")

lines.append("")
# Specific conclusion about db_011
if 'db_011' in breached_pairs['cmdb_id'].values:
    db11 = breached_pairs[breached_pairs['cmdb_id']=='db_011'].sort_values('severity', ascending=False)
    db11_lines = []
    for _, r in db11.iterrows():
        db11_lines.append(f"{r['kpi_name']} (severity={float(r['severity']):.3f}, earliest={pd.to_datetime(r['earliest_breach_timestamp']).tz_convert(tz).isoformat()})")
    lines.append("Notable: db_011 shows the most extreme spikes and is not unique in having breaches, but its severities are far higher than others (examples: " + "; ".join(db11_lines) + ").")
else:
    lines.append("db_011 did not breach these KPIs in the window.")

lines.append("")
lines.append("Overall conclusion: I/O/write spikes are widespread across multiple DB nodes (several db_* show breaches), but db_011 exhibits by far the largest spikes (very high severities for both LFParaWrite_Per_Sec and LFSync_Per_Sec). So the phenomenon is not exclusive to db_011, but db_011 is the most severely affected.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id             kpi_name  breached earliest_breach_timestamp  max_value_in_window  global_p95    severity
0   db_011  LFParaWrite_Per_Sec      True 2020-05-31 02:51:00+08:00                17.89      0.1400  126.785714
1   db_011       LFSync_Per_Sec      True 2020-05-31 02:37:00+08:00                37.99      0.4510   83.235033
2   db_003       LFSync_Per_Sec      True 2020-05-31 02:34:00+08:00                 9.10      3.3480    1.718041
3   db_003  LFParaWrite_Per_Sec      True 2020-05-31 02:34:00+08:00                 3.98      1.8930    1.102483
4   db_002       LFSync_Per_Sec      True 2020-05-31 02:41:00+08:00                 1.18      0.5840    1.020548
..     ...                  ...       ...                       ...                  ...         ...         ...
21  db_008  LFParaWrite_Per_Sec     False                       NaT                 2.26      2.4505   -0.077739
22  db_013  LFParaWrite_Per_Sec     False                       NaT                 0.20      0.2240   -0.107143
23  db_012       LFSync_Per_Sec     False                       NaT                 0.50      0.5625   -0.111111
24  db_008       LFSync_Per_Sec     False                       NaT                 3.89      4.6720   -0.167380
25  db_010  LFParaWrite_Per_Sec     False                       NaT                 0.09      0.1345   -0.330855

[26 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```